# 09 — Threshold recheck

Every threshold in `src/rules.py` was measured in notebook 04 or in the transcription job,
under preprocessing and decoding that differ from what `CareVoicePipeline` actually runs.
This notebook measures whether those numbers survive. They are quoted in the paper, so
this has to pass first.

| § | Question | Threshold at stake |
|---|---|---|
| 0 | Which acoustic window did notebook 04 use? | all acoustic |
| 1 | Does the notebook-04 recipe reproduce the saved training features? | (sanity) |
| 2 | **Level:** RMS 0.05 (notebook 04) vs −23 LUFS (`src/preprocess.py`) | `AROUSAL_HIGH` 0.105, `DISTRESS_HIGH` 0.0677 |
| 3 | **Padding:** does zero-padding to 4 s drive the distress score? | `DISTRESS_HIGH` |
| 4 | **ASR:** re-derive `ASR_MIN` with the pipeline's exact decoder | `ASR_MIN` −0.4934 |
| 5 | Speech guard on real groans; `no_speech_prob` as an alternative guard | `SPEECH_GUARD` |

All audio goes through `src.preprocess`, and all embeddings and transcripts through
`src.pipeline.embed` / `src.pipeline.transcribe`, the same functions `process()` calls.
The one exception is §0–2's **notebook-04 recipe**: it is copied verbatim and labelled,
because reproducing it is the point of the comparison.

**Inputs (Kaggle).** Attach:
- `paultimothymooney/medical-speech-transcription-and-intent`
- a private dataset with this repo's `src/`, `models/arousal_head.joblib`,
  `models/distress_gate.joblib`, `models/X_vivae_aug.npy`, `results/transcripts.csv`,
  `notebooks/04_acoustic_branch.ipynb` (optionally `models/clinicalbert_requestclass/`
  for text accuracy, and `models/faster-whisper-small/` + `models/wavlm-base-plus/`)

Internet on (VIVAE comes from Zenodo; Whisper/WavLM from the Hub if not attached).
CPU is enough: Whisper must run int8 on CPU anyway to match the pipeline.

Outputs go to `results/threshold_recheck_*.csv|json|png`.

In [ ]:
import os, sys, re, glob, json, time, random, subprocess, tempfile, zipfile, urllib.request
from pathlib import Path

for pkg, mod in [("faster-whisper", "faster_whisper"), ("jiwer", "jiwer")]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

ON_KAGGLE = Path("/kaggle/input").exists()

def find_repo():
    # A folder holding src/pipeline.py: this repo locally, or the attached code dataset.
    cands = [Path.cwd(), Path.cwd().parent]
    if ON_KAGGLE:
        cands += [p.parent.parent for p in Path("/kaggle/input").glob("**/src/pipeline.py")]
    for c in cands:
        if (c / "src" / "pipeline.py").exists():
            return c.resolve()
    raise FileNotFoundError("src/pipeline.py not found; attach the code dataset")

REPO = find_repo()
sys.path.insert(0, str(REPO))

CONFIG = {
    "seed": 42,
    # Medical clips for the level test and the ASR recheck (same clips for both).
    "n_med": int(os.environ.get("CV09_N_MED", 200)),
    # Clips used to confirm the notebook-04 recipe reproduces X_vivae_aug.npy.
    "n_repro": int(os.environ.get("CV09_N_REPRO", 48)),
    "med_dir": os.environ.get("CV09_MED_DIR", "/kaggle/input"),
    # Same root path as notebook 04, so sorted() order and therefore degrade() seeds match.
    "vivae_dir": os.environ.get("CV09_VIVAE_DIR", "/kaggle/working/vivae"),
    "vivae_url": "https://zenodo.org/records/4066235/files/VIVAE.zip?download=1",
    "out_dir": Path(os.environ.get("CV09_OUT",
                    "/kaggle/working/results" if ON_KAGGLE else str(REPO / "results"))),
    # Matches the demo machine, where DeepFilterNet is not installed. If you install it
    # for the demo, set True and rerun: the thresholds must be measured as deployed.
    "use_denoise": False,
    "positives": ["pain", "fear"],          # distress gate positives (notebook 04)
    "zero_lengths_s": [0.5, 1.0, 2.0, 4.0],
}
CONFIG["out_dir"].mkdir(parents=True, exist_ok=True)

# Load models from the repo's models/ when present, so nothing touches the network.
LOCAL_WHISPER = REPO / "models" / "faster-whisper-small"
LOCAL_WAVLM = REPO / "models" / "wavlm-base-plus"
os.environ["HF_HUB_OFFLINE"] = "1" if (LOCAL_WHISPER.exists() and LOCAL_WAVLM.exists()) else "0"

import numpy as np, pandas as pd, torch, joblib, librosa, jiwer, soundfile as sf
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, wilcoxon

import src.preprocess as pp
from src.preprocess import load_audio, denoise, trim, pad_or_crop, speech_present, SPEECH_GUARD
from src.pipeline import CONFIG as PIPE, transcribe, embed
from src.rules import CONFIG as RULES

pp.CONFIG["enable_denoise"] = CONFIG["use_denoise"]

random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
SR = PIPE["sr"]
WIN = int(round(PIPE["acoustic_window_s"] * SR))
THR = {"arousal": RULES["AROUSAL_HIGH"], "distress": RULES["DISTRESS_HIGH"]}
OUT = CONFIG["out_dir"]
print(f"repo={REPO}  on_kaggle={ON_KAGGLE}  seed={CONFIG['seed']}")
print(f"window={PIPE['acoustic_window_s']} s  thresholds={THR}  ASR_MIN={RULES['ASR_MIN']}")
print("decoder:", PIPE["whisper_decode"])

## 0. Which window did notebook 04 use?

Checked first: if notebook 04 used a different window from the pipeline, that is a
second preprocessing mismatch and changes how every number below is read.

In [ ]:
nb04 = json.loads((REPO / "notebooks" / "04_acoustic_branch.ipynb").read_text())
src04 = ["".join(c["source"]) for c in nb04["cells"]]
hits = [(i, l.strip()) for i, s in enumerate(src04) for l in s.splitlines()
        if re.search(r"WIN_S\s*=|np\.pad\(|\[:WIN\]|extract_aug\(|def degrade|w = w \+ r\.normal", l)]
for i, l in hits:
    print(f"  cell {i:2d}: {l}")

win04 = float(re.search(r"WIN_S\s*=\s*16000,\s*([\d.]+)", src04[1]).group(1))
print(f"\nnotebook 04 window: {win04} s, crop from start, zero-pad at end")
print(f"pipeline window   : {PIPE['acoustic_window_s']} s, crop from start, zero-pad at end")
assert win04 == PIPE["acoustic_window_s"], "WINDOW MISMATCH: second preprocessing mismatch"
print('''
Same window and same pad position: train/inference are consistent in WINDOW.
They are NOT consistent in what fills the padding. Notebook 04 trained both heads on
degrade(load_audio(p)), and degrade() adds Gaussian noise to the whole 4 s window after
padding. So in training the padded region was noise, never digital zeros.
  - VIVAE (gate positives + negatives) and CREMA-D (arousal head, gate negatives): noise-filled pad
  - X_med (the clips the false-alarm rates were measured on): clean, ZERO pad
  - live inference: clean, ZERO pad
Section 3 measures what that does.''')

## Models and data

In [ ]:
from faster_whisper import WhisperModel
from transformers import AutoFeatureExtractor, WavLMModel

DEV = "cuda" if torch.cuda.is_available() else "cpu"   # WavLM only, fp32 like the pipeline
asr = WhisperModel(str(LOCAL_WHISPER) if LOCAL_WHISPER.exists() else "Systran/faster-whisper-small",
                   device="cpu", compute_type=PIPE["whisper_compute_type"])   # exactly as deployed
wavlm_src = str(LOCAL_WAVLM) if LOCAL_WAVLM.exists() else "microsoft/wavlm-base-plus"
fe = AutoFeatureExtractor.from_pretrained(wavlm_src)
wavlm = WavLMModel.from_pretrained(wavlm_src).to(DEV).eval()
for p in wavlm.parameters():
    p.requires_grad = False

arousal_b = joblib.load(REPO / "models" / "arousal_head.joblib")
gate_b = joblib.load(REPO / "models" / "distress_gate.joblib")
FEAR = arousal_b["fear_index"]
POS = list(gate_b["model"].classes_).index(1)

def heads(X):
    X = np.atleast_2d(X)
    return (arousal_b["model"].predict_proba(X)[:, FEAR],
            gate_b["model"].predict_proba(X)[:, POS])

bert_dir = REPO / "models" / "clinicalbert_requestclass"
bert = None
if bert_dir.exists():
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(bert_dir)
    bert = AutoModelForSequenceClassification.from_pretrained(bert_dir).eval()
    BERT_CLASSES = json.loads((bert_dir / "classes.json").read_text())["classes"]
print("WavLM on", DEV, "| ClinicalBERT:", "yes" if bert else "not attached, text accuracy skipped")

In [ ]:
# ---- medical: the same rows that produced transcripts.csv, seeded sample ----
tr = pd.read_csv(REPO / "results" / "transcripts.csv")
idx = {}
for p in sorted(glob.glob(f"{CONFIG['med_dir']}/**/*.wav", recursive=True)):
    if "vivae" not in p.lower():
        idx.setdefault(os.path.basename(p), p)       # the zip extracts two copies; keep one
tr["local_path"] = tr.file_name.map(idx)
print(f"transcripts.csv rows {len(tr)}, audio found for {tr.local_path.notna().sum()}")
med = (tr.dropna(subset=["local_path"])
         .sample(n=min(CONFIG["n_med"], tr.local_path.notna().sum()), random_state=CONFIG["seed"])
         .reset_index(drop=True))
print(f"medical sample: {len(med)} clips, {med.speaker_id.nunique()} speakers")

# ---- VIVAE: identical discovery to notebook 04 so the index-based degrade() seeds match ----
DST = CONFIG["vivae_dir"]
if not glob.glob(f"{DST}/**/*.wav", recursive=True):
    os.makedirs(DST, exist_ok=True)
    urllib.request.urlretrieve(CONFIG["vivae_url"], f"{DST}.zip")
    with zipfile.ZipFile(f"{DST}.zip") as z:
        z.extractall(DST)
vv = sorted(glob.glob(f"{DST}/**/*.wav", recursive=True))
rows = []
for i, p in enumerate(vv):
    spk, emo, inten, item = os.path.splitext(os.path.basename(p))[0].split("_")
    rows.append({"nb04_index": i, "path": p, "file": os.path.basename(p), "subset": Path(p).parent.name,
                 "speaker": spk, "emotion": emo, "intensity": inten,
                 "duration_s": sf.info(p).duration})
vd_all = pd.DataFrame(rows)
print(f"\nVIVAE files: {len(vd_all)} (X_vivae_aug.npy rows: {len(np.load(REPO/'models'/'X_vivae_aug.npy'))})")
print(vd_all.groupby("subset").size().to_string())
dups = vd_all.file.duplicated(keep=False).sum()
print(f"files whose name appears more than once: {dups}")
if dups:
    print("  -> the zip holds overlapping sets; notebook 04 trained the gate on the duplicates too.")
    print("     Speaker-grouped CV keeps copies in one fold (no leakage) but they are double-weighted.")
vd = vd_all.drop_duplicates("file").reset_index(drop=True)      # unique clips for analysis
vpos = vd[vd.emotion.isin(CONFIG["positives"])].reset_index(drop=True)
print(f"unique clips {len(vd)}; positives (pain+fear) {len(vpos)}: "
      f"{vpos.emotion.value_counts().to_dict()}")

## Recipes

- **A — notebook-04 recipe** (copied verbatim, *not* `src/preprocess.py`):
  `librosa.load` → RMS 0.05 (gain capped ×10) → 4 s window. This is what the
  thresholds were fitted under.
- **B — `src.preprocess.load_audio`**: −23 LUFS → 4 s window. Isolates the level change.
- **C — the pipeline**: `load_audio` → `denoise` → `trim` → 4 s window. What `process()`
  actually feeds WavLM.

All three are embedded with `src.pipeline.embed`.

In [ ]:
from scipy.signal import butter, lfilter

# ---- notebook 04, cell 1: load_audio (verbatim apart from the window split) ----
TARGET_RMS = 0.05
def nb04_level(path):
    wav, _ = librosa.load(path, sr=SR, mono=True)
    rms = np.sqrt(np.mean(wav**2)) if len(wav) else 0.0
    if rms > 1e-8:
        wav = wav * min(TARGET_RMS / rms, 10.0)
        peak = np.abs(wav).max()
        if peak > 0.99:
            wav = wav * (0.99 / peak)
    return wav.astype(np.float32)

def nb04_load(path):                                  # == notebook 04's load_audio
    return pad_or_crop(nb04_level(path), WIN).astype(np.float32)

# ---- notebook 04, cell 3: degrade (verbatim) ----
def degrade(wav, sr=16000, seed=0):
    r = np.random.default_rng(seed)
    cutoff = r.uniform(3000, 7000)
    b, a = butter(6, cutoff / (sr/2), btype="low")
    w = lfilter(b, a, wav)
    snr = r.uniform(5, 25)
    sp = np.mean(w**2)
    if sp > 1e-10:
        w = w + r.normal(0, np.sqrt(sp / (10**(snr/10))), len(w))
    w = w * r.uniform(0.5, 1.5)
    peak = np.abs(w).max()
    if peak > 0.99:
        w = w * (0.99 / peak)
    return w.astype(np.float32)

def recipe_B(path):
    return load_audio(path, target_sr=SR)

def recipe_C(path):
    wav, meta = load_audio(path, target_sr=SR, return_meta=True)
    return trim(denoise(wav, sr=SR), sr=SR)

def emb(wav):
    return embed(fe, wavlm, wav, DEV)[0]

@torch.no_grad()
def emb_native(wav):
    # Same model and pooling as embed(), but no 4 s window: the clip at its own length.
    x = fe(wav.astype(np.float32), sampling_rate=SR, return_tensors="pt").input_values.to(DEV)
    return wavlm(x).last_hidden_state.mean(1).float().cpu().numpy()[0]

def fill(wav):
    # Tile a short clip / centre-crop a long one, so the window is all signal.
    if len(wav) < WIN:
        return np.resize(wav, WIN).astype(np.float32)
    off = (len(wav) - WIN) // 2
    return wav[off:off + WIN].astype(np.float32)

def run(fn, items, tag):
    out, t0 = [], time.time()
    for k, it in enumerate(items):
        out.append(fn(it))
        if (k + 1) % 100 == 0 or k + 1 == len(items):
            print(f"  {tag}: {k+1}/{len(items)}  {time.time()-t0:.0f}s")
    return np.vstack(out)

## 1. Sanity: does recipe A reproduce the saved training features?

Re-extract the first `n_repro` VIVAE clips exactly as notebook 04 did
(`degrade(load_audio(p), seed=index)`) and compare with `models/X_vivae_aug.npy`.
Notebook 04 ran WavLM in fp16 on GPU, so expect cosine ≈ 0.99+, not 1.0. If this fails,
recipe A is not what the heads were trained on and §2–3 compare against the wrong baseline.

In [ ]:
X_saved = np.load(REPO / "models" / "X_vivae_aug.npy")
probe = vd_all.head(CONFIG["n_repro"])
X_re = run(lambda r: emb(degrade(nb04_load(r.path), seed=int(r.nb04_index))),
           list(probe.itertuples()), "repro")
Xs = X_saved[probe.nb04_index.values]
cos = (X_re * Xs).sum(1) / (np.linalg.norm(X_re, axis=1) * np.linalg.norm(Xs, axis=1))
d_re, d_sv = heads(X_re)[1], heads(Xs)[1]
print(f"cosine to saved features: min {cos.min():.4f}  median {np.median(cos):.4f}")
print(f"gate score |re - saved|: median {np.median(abs(d_re-d_sv)):.2e}  max {abs(d_re-d_sv).max():.2e}")
REPRO_OK = bool(np.median(cos) > 0.99)
print("recipe A reproduces training:", REPRO_OK)

## 2. Level: RMS 0.05 vs −23 LUFS

200 medical clips (the false-alarm side) and every unique VIVAE pain/fear clip (the
recall side), embedded under A, B and C, both heads scored on all three.

Read the **crossing rates**, not the means. Both thresholds sit near the bottom of the
score range where the curve is steep; what matters is how many clips change side.

Caveat: the gate was *fitted* on these VIVAE clips (augmented), so absolute VIVAE recall
here is in-sample and optimistic. The A-vs-B-vs-C *shift* is the measurement.

In [ ]:
sets = {"medical": med.local_path.tolist(), "vivae_pos": vpos.path.tolist()}
recipes = {"A_nb04": nb04_load, "B_lufs": recipe_B, "C_pipeline": recipe_C}
feats = {}
for s, paths in sets.items():
    for r, fn in recipes.items():
        feats[(s, r)] = run(lambda p: emb(fn(p)), paths, f"{s}/{r}")
        np.save(OUT / f"threshold_recheck_X_{s}_{r}.npy", feats[(s, r)])

scores = []
for (s, r), X in feats.items():
    a, d = heads(X)
    ids = med.file_name.tolist() if s == "medical" else vpos.file.tolist()
    scores += [{"set": s, "recipe": r, "clip": c, "arousal": ai, "distress": di}
               for c, ai, di in zip(ids, a, d)]
level = pd.DataFrame(scores)
level.to_csv(OUT / "threshold_recheck_level_scores.csv", index=False)

rows = []
for s in sets:
    for head in ("arousal", "distress"):
        wide = level[level.set == s].pivot(index="clip", columns="recipe", values=head)
        base = wide["A_nb04"] >= THR[head]
        for r in ("A_nb04", "B_lufs", "C_pipeline"):
            above = wide[r] >= THR[head]
            rows.append({"set": s, "head": head, "recipe": r, "threshold": THR[head],
                         "median": wide[r].median(), "frac_above": above.mean(),
                         "flips_vs_A": int((above != base).sum()),
                         "up_vs_A": int((above & ~base).sum()),
                         "down_vs_A": int((~above & base).sum()), "n": len(wide)})
level_tbl = pd.DataFrame(rows)
level_tbl.to_csv(OUT / "threshold_recheck_level.csv", index=False)
print("frac_above = false-alarm rate on 'medical', in-sample recall on 'vivae_pos'\n")
print(level_tbl.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for col, head in enumerate(("arousal", "distress")):
    for row, other in enumerate(("B_lufs", "C_pipeline")):
        ax = axes[row, col]
        for s, c in (("medical", "tab:blue"), ("vivae_pos", "tab:red")):
            w = level[level.set == s].pivot(index="clip", columns="recipe", values=head)
            ax.scatter(w["A_nb04"].clip(1e-12), w[other].clip(1e-12), s=9, alpha=.5, c=c, label=s)
        ax.set_xscale("log"); ax.set_yscale("log")
        lim = [1e-12, 1]
        ax.plot(lim, lim, "k:", lw=.8)
        ax.axvline(THR[head], c="gray", ls="--", lw=.8); ax.axhline(THR[head], c="gray", ls="--", lw=.8)
        ax.set_xlabel(f"{head}: A (notebook 04, RMS 0.05)"); ax.set_ylabel(f"{head}: {other}")
        ax.set_title(f"{head}: A vs {other}  (dashed = threshold)")
        ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "threshold_recheck_level.png", dpi=140); plt.show()

## 3. Padding

Zeros push the gate positive (0.9996 on a silent clip). The 4 s window zero-pads every
short clip, and a VIVAE groan is about 1 s. So the gate score could partly measure how
short a clip is.

- **3a** the gate on pure zeros, at native length and through the 4 s window
- **3b** each VIVAE pain/fear clip: training-exact (noise-filled pad) vs clean zero-pad
  vs clean tile/centre-crop vs native length
- **3c** duration vs distress score across all unique VIVAE clips, and across the
  medical sample

If zero-pad scores sit clearly above fill scores, and duration correlates negatively with
score under zero-pad but not under fill, padding drives part of the score.

In [ ]:
zr = []
for L in CONFIG["zero_lengths_s"]:
    z = np.zeros(int(L * SR), dtype=np.float32)
    for mode, e in (("native_length", emb_native(z)), ("4s_window", emb(z))):
        a, d = heads(e)
        zr.append({"zeros_s": L, "mode": mode, "arousal": a[0], "distress": d[0]})
# Low-level noise: raw into WavLM, and through the pipeline path (load_audio normalises it
# to -23 LUFS, then trim). Only the pipeline_path rows describe deployed behaviour.
with tempfile.TemporaryDirectory() as tmp:
    for db in (-60, -50, -40):
        n = (np.random.default_rng(CONFIG["seed"]).normal(0, 10**(db/20), WIN)).astype(np.float32)
        p = os.path.join(tmp, f"n{db}.wav"); sf.write(p, n, SR)
        for mode, e in (("raw_4s_window", emb(n)), ("pipeline_path", emb(recipe_C(p)))):
            a, d = heads(e)
            zr.append({"zeros_s": f"4 s Gaussian {db} dBFS", "mode": mode, "arousal": a[0], "distress": d[0]})
zeros_tbl = pd.DataFrame(zr)
zeros_tbl.to_csv(OUT / "threshold_recheck_zeros.csv", index=False)
print(zeros_tbl.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\n(4s_window rows are identical by construction: shorter zeros are padded to 4 s of zeros.)")
print("Pure zeros never reach the gate in deployment (the speech guard stops them); noise above")
print("the RMS floor does, so the pipeline_path noise rows are the deployment-relevant ones.")

In [ ]:
variants = {
    "train_exact_noisepad": lambda r: emb(degrade(nb04_load(r.path), seed=int(r.nb04_index))),
    "clean_zeropad":        lambda r: emb(nb04_load(r.path)),
    "clean_fill":           lambda r: emb(fill(nb04_level(r.path))),
    "clean_native":         lambda r: emb_native(nb04_level(r.path)),
}
pad = vpos[["file", "emotion", "intensity", "duration_s", "nb04_index"]].copy()
for k, fn in variants.items():
    pad[k] = heads(run(fn, list(vpos.itertuples()), k))[1]
pad.to_csv(OUT / "threshold_recheck_padding.csv", index=False)

rows = []
for k in variants:
    rows.append({"variant": k, "median_distress": pad[k].median(),
                 "frac_above_thr": (pad[k] >= THR["distress"]).mean()})
pad_tbl = pd.DataFrame(rows)
short = pad.duration_s < PIPE["acoustic_window_s"]
diff = pad.loc[short, "clean_zeropad"] - pad.loc[short, "clean_fill"]
wx = wilcoxon(diff) if short.sum() > 10 else None
print(pad_tbl.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nclips shorter than the window: {short.sum()}/{len(pad)}")
print(f"zero-pad minus fill, short clips: median {diff.median():+.4f}, "
      f"zero-pad higher in {(diff > 0).mean():.0%}" + (f", Wilcoxon p={wx.pvalue:.2e}" if wx else ""))
crossings = ((pad.clean_zeropad >= THR["distress"]) != (pad.clean_fill >= THR["distress"])).sum()
print(f"clips on different sides of DISTRESS_HIGH under zero-pad vs fill: {crossings}")

In [ ]:
# 3c: duration vs score, all unique VIVAE clips (clean zero-pad and clean fill) + medical
dur_feats = {k: run(variants[k], list(vd.itertuples()), f"all-vivae/{k}")
             for k in ("clean_zeropad", "clean_fill")}
vd_scores = vd[["file", "emotion", "duration_s"]].copy()
for k, X in dur_feats.items():
    vd_scores[k] = heads(X)[1]
vd_scores.to_csv(OUT / "threshold_recheck_vivae_duration.csv", index=False)

med_dur = np.array([sf.info(p).duration for p in med.local_path])
med_d = level[(level.set == "medical") & (level.recipe == "A_nb04")].set_index("clip").loc[med.file_name, "distress"].values

rows = []
for name, sub in (("all VIVAE", vd_scores), ("VIVAE pain+fear", vd_scores[vd_scores.emotion.isin(CONFIG["positives"])]),
                  ("VIVAE other", vd_scores[~vd_scores.emotion.isin(CONFIG["positives"])])):
    for k in ("clean_zeropad", "clean_fill"):
        rho, p = spearmanr(sub.duration_s, sub[k])
        rows.append({"set": name, "variant": k, "n": len(sub), "spearman_rho": rho, "p": p})
rho, p = spearmanr(med_dur, med_d)
rows.append({"set": "medical sample", "variant": "clean_zeropad (A)", "n": len(med), "spearman_rho": rho, "p": p})
dur_tbl = pd.DataFrame(rows)
dur_tbl.to_csv(OUT / "threshold_recheck_duration_corr.csv", index=False)
print(dur_tbl.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, k in zip(ax, ("clean_zeropad", "clean_fill")):
    for pos, c in ((True, "tab:red"), (False, "tab:gray")):
        m = vd_scores.emotion.isin(CONFIG["positives"]) == pos
        a.scatter(vd_scores.duration_s[m], vd_scores[k][m].clip(1e-6), s=8, alpha=.5, c=c,
                  label="pain+fear" if pos else "other")
    a.axhline(THR["distress"], c="k", ls="--", lw=.8); a.set_yscale("log")
    a.set_xlabel("clip duration (s)"); a.set_title(f"distress vs duration, {k}")
ax[0].set_ylabel("distress score"); ax[0].legend()
plt.tight_layout(); plt.savefig(OUT / "threshold_recheck_padding.png", dpi=140); plt.show()

## 4. ASR: re-derive `ASR_MIN` with the deployed decoder

Same medical clips as §2, through the pipeline path (`load_audio` → `denoise` → `trim`)
and `src.pipeline.transcribe`: faster-whisper int8 on CPU, beam 5, `temperature=0.0`
(no fallback ladder), `condition_on_previous_text=False`, `vad_filter=False`,
log-prob = token-weighted mean of segment `avg_logprob`. All of that comes from
`src.pipeline.CONFIG`, so it cannot drift from what the app runs.

WER uses the normalisation that reproduces `transcripts.csv`'s `ref` column (lowercase,
every non-alphanumeric character to a space; 99.9% row match).

Per-segment `no_speech_prob`, `temperature` and `compression_ratio` are logged too.

In [ ]:
def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(s).lower())).strip()

def asr_row(path):
    wav, meta = load_audio(path, target_sr=SR, return_meta=True)
    wav = trim(denoise(wav, sr=SR), sr=SR)
    ok, why = speech_present(meta, len(wav), sr=SR)
    t0 = time.perf_counter()
    text, lp, segs = transcribe(asr, wav)
    return {"raw_rms": meta["raw_rms"], "trimmed_s": len(wav) / SR,
            "speech_present": ok, "speech_reason": why, "transcript": text, "asr_logprob": lp,
            "n_segments": len(segs),
            "no_speech_probs": json.dumps([round(s.no_speech_prob, 5) for s in segs]),
            "no_speech_prob_max": max((s.no_speech_prob for s in segs), default=np.nan),
            "temperatures": json.dumps(sorted({s.temperature for s in segs})),
            "compression_ratio_max": max((s.compression_ratio for s in segs), default=np.nan),
            "asr_ms": (time.perf_counter() - t0) * 1000}

asr_med = pd.DataFrame([asr_row(p) for p in med.local_path])
asr_med = pd.concat([med[["file_name", "phrase", "prompt", "req_class", "asr_logprob", "hyp", "row_wer"]]
                       .rename(columns={"asr_logprob": "old_logprob", "hyp": "old_hyp", "row_wer": "old_wer"}),
                     asr_med.rename(columns={"asr_logprob": "new_logprob"})], axis=1)
asr_med["ref"] = asr_med.phrase.map(norm)
asr_med["new_hyp"] = asr_med.transcript.map(norm)
asr_med["new_wer"] = [jiwer.wer(r, h) if r else np.nan for r, h in zip(asr_med.ref, asr_med.new_hyp)]

if bert is not None:
    @torch.no_grad()
    def classify(t):
        enc = tok(t, return_tensors="pt", truncation=True, max_length=PIPE["bert_max_length"])
        return BERT_CLASSES[int(bert(**enc).logits.argmax(-1))]
    asr_med["new_pred"] = asr_med.transcript.map(classify)
    asr_med["new_correct"] = asr_med.new_pred == asr_med.req_class

temps = set().union(*[set(json.loads(t)) for t in asr_med.temperatures])
assert temps <= {0.0}, f"fallback ladder fired: temperatures {temps}"
print(f"all segments decoded at temperature 0.0 | median ASR {asr_med.asr_ms.median():.0f} ms/clip")
print(f"guard rejected {(~asr_med.speech_present).sum()} of {len(asr_med)} medical clips")

In [ ]:
q = 0.20
new_min = float(asr_med.new_logprob.quantile(q))
old_min_sample = float(asr_med.old_logprob.quantile(q))
print(f"ASR_MIN in rules.py (old decoder, full corpus n=6661) : {RULES['ASR_MIN']:.4f}")
print(f"old decoder, 20th pct on this sample (n={len(asr_med)})       : {old_min_sample:.4f}   <- sampling noise reference")
print(f"NEW decoder, 20th pct on this sample                     : {new_min:.4f}")
paired = asr_med.new_logprob - asr_med.old_logprob
print(f"paired shift new-old: median {paired.median():+.4f}, Spearman rho {spearmanr(asr_med.new_logprob, asr_med.old_logprob)[0]:.3f}")
print(f"WER (corpus-level on sample): old {jiwer.wer(asr_med.ref.tolist(), asr_med.old_hyp.fillna('').tolist()):.3f}"
      f"  new {jiwer.wer(asr_med.ref.tolist(), asr_med.new_hyp.tolist()):.3f}")

def side_table(thr):
    rows = []
    for name, m in (("below", asr_med.new_logprob < thr), ("above", asr_med.new_logprob >= thr)):
        r = {"side": name, "n": int(m.sum()), "frac_wer_gt_0.5": (asr_med.new_wer[m] > 0.5).mean(),
             "mean_wer": asr_med.new_wer[m].mean()}
        if bert is not None:
            r["text_accuracy"] = asr_med.new_correct[m].mean()
        rows.append(r)
    return pd.DataFrame(rows)

print("\nsplit at the NEW threshold (rules.py claims 18.3%/0.707 below vs 1.5%/0.86 above):")
asr_sides = side_table(new_min)
print(asr_sides.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
asr_med.to_csv(OUT / "threshold_recheck_asr.csv", index=False)
asr_sides.to_csv(OUT / "threshold_recheck_asr_sides.csv", index=False)

## 5. Speech guard on real groans, and `no_speech_prob` as an alternative

The regression check that matters: the guard must never swallow a genuine wordless cry.
Every VIVAE pain/fear clip goes through the pipeline path and `speech_present` with the
current (placeholder) `SPEECH_GUARD`. The negative controls are run the same way.

`no_speech_prob` is logged for both. If it separates silence from medical speech **and**
from groans as well as the RMS floor does, it is a decoder-native alternative guard. A
groan is not speech, so expect it to struggle on exactly the case the guard must protect.

In [ ]:
asr_v = pd.concat([vpos[["file", "emotion", "intensity", "duration_s"]].reset_index(drop=True),
                   pd.DataFrame([asr_row(p) for p in vpos.path])], axis=1)

negs = {"zeros_5ms": np.zeros(int(0.005 * SR)), "zeros_0.5s": np.zeros(SR // 2),
        "zeros_2s": np.zeros(2 * SR)}
for db in (-60, -50, -40):
    negs[f"gauss_{db}dBFS_2s"] = np.random.default_rng(abs(db)).normal(0, 10**(db/20), 2 * SR)
neg_rows = []
with tempfile.TemporaryDirectory() as tmp:
    for name, sig in negs.items():
        p = os.path.join(tmp, f"{name}.wav"); sf.write(p, sig.astype(np.float32), SR)
        neg_rows.append({"control": name, **asr_row(p)})
asr_neg = pd.DataFrame(neg_rows)

asr_v.to_csv(OUT / "threshold_recheck_guard_vivae.csv", index=False)
asr_neg.to_csv(OUT / "threshold_recheck_guard_negatives.csv", index=False)

print(f"SPEECH_GUARD (placeholders): {SPEECH_GUARD}")
print(f"VIVAE pain/fear passing the guard: {asr_v.speech_present.sum()}/{len(asr_v)}")
if (~asr_v.speech_present).any():
    print("  REJECTED GROANS (guard too strict):")
    print(asr_v.loc[~asr_v.speech_present, ["file", "duration_s", "raw_rms", "trimmed_s", "speech_reason"]].to_string(index=False))
print(f"medical clips passing: {asr_med.speech_present.sum()}/{len(asr_med)}")
print("\nnegative controls:")
print(asr_neg[["control", "raw_rms", "trimmed_s", "speech_present", "transcript", "no_speech_prob_max"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

def summ(x):
    x = pd.Series(x).dropna()
    return f"n={len(x)} min={x.min():.3f} p05={x.quantile(.05):.3f} median={x.median():.3f} max={x.max():.3f}" if len(x) else "no segments"
print("\nno_speech_prob (max over segments):")
print("  medical      ", summ(asr_med.no_speech_prob_max))
print("  VIVAE groans ", summ(asr_v.no_speech_prob_max), f"| no segments at all: {(asr_v.n_segments == 0).sum()}")
print("  negatives    ", summ(asr_neg.no_speech_prob_max), f"| no segments at all: {(asr_neg.n_segments == 0).sum()}")

## Summary

Written to `results/threshold_recheck.json`. The verdict fields are computed, not
asserted; read the tables above before quoting any of them.

In [ ]:
def lt(set_, head, recipe, col):
    return float(level_tbl.query("set==@set_ and head==@head and recipe==@recipe")[col].iloc[0])

summary = {
    "seed": CONFIG["seed"], "n_medical": len(med), "n_vivae_pos": len(vpos),
    "denoise_active": CONFIG["use_denoise"],
    "window": {"nb04_s": win04, "pipeline_s": PIPE["acoustic_window_s"], "same": win04 == PIPE["acoustic_window_s"],
               "pad_content_training": "Gaussian noise (degrade after pad)", "pad_content_inference": "zeros"},
    "recipe_A_reproduces_training": {"ok": REPRO_OK, "median_cosine": float(np.median(cos)), "n": len(cos)},
    "level": {f"{s}/{h}": {r: {"frac_above": lt(s, h, r, "frac_above"), "flips_vs_A": int(lt(s, h, r, "flips_vs_A"))}
                           for r in ("A_nb04", "B_lufs", "C_pipeline")}
              for s in sets for h in ("arousal", "distress")},
    "padding": {"zeros": zeros_tbl.to_dict("records"),
                "vivae_pos_by_variant": pad_tbl.to_dict("records"),
                "zeropad_minus_fill_median_short_clips": float(diff.median()),
                "wilcoxon_p": float(wx.pvalue) if wx else None,
                "threshold_crossings_zeropad_vs_fill": int(crossings),
                "duration_spearman": dur_tbl.to_dict("records")},
    "asr": {"ASR_MIN_rules_py": RULES["ASR_MIN"], "old_p20_on_sample": old_min_sample,
            "new_p20_on_sample": new_min, "paired_shift_median": float(paired.median()),
            "decoder": PIPE["whisper_decode"], "compute_type": PIPE["whisper_compute_type"],
            "sides_at_new_threshold": asr_sides.to_dict("records")},
    "guard": {"SPEECH_GUARD": SPEECH_GUARD, "placeholders": True,
              "vivae_pos_pass": int(asr_v.speech_present.sum()), "vivae_pos_n": len(asr_v),
              "medical_pass": int(asr_med.speech_present.sum()), "medical_n": len(asr_med),
              "negatives_rejected": int((~asr_neg.speech_present).sum()), "negatives_n": len(asr_neg),
              "no_speech_prob_max": {"medical_min": float(asr_med.no_speech_prob_max.min()),
                                     "vivae_pos_min": float(asr_v.no_speech_prob_max.min()) if asr_v.no_speech_prob_max.notna().any() else None,
                                     "negatives_max": float(asr_neg.no_speech_prob_max.max()) if asr_neg.no_speech_prob_max.notna().any() else None}},
}
json.dump(summary, open(OUT / "threshold_recheck.json", "w"), indent=2, default=float)
print(json.dumps({k: summary[k] for k in ("window", "recipe_A_reproduces_training", "asr")}, indent=2, default=float))
print("\nwritten:", sorted(p.name for p in OUT.glob("threshold_recheck*")))